# Сравнение моделей

Train: 24 004 строки. Пять общих блоков StratifiedGroupKFold, seed=42. Одинаковые профили остаются вместе. Преобразования обучаются внутри блока. Параметры фиксированы, early stopping отключён. Test не используем.

| Набор | Признаки |
|---|---|
| raw | Исходные без ID и цели |
| base | raw, возрастная группа, задержки, нагрузка, изменения сумм |
| sequence | base и пять признаков последовательности задержек |
| monthly | base и 13 признаков месячных сумм |

CatBoost получает категории строками. HGB и XGBoost используют one-hot.

ROC-AUC и AP лучше при больших значениях, Brier при меньших. Recall@20% показывает долю дефолтов среди 20% клиентов с наибольшим риском. При равных рисках на границе выбор усредняется без использования ответов.

In [ ]:
from pathlib import Path
import pandas as pd

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
folder = root / 'reports/experiments/seed_42'
metrics = ['ROC-AUC', 'AP', 'Brier', 'Recall@20%', 'Precision@20%']
folds = pd.read_csv(folder / 'fold_metrics.csv')
summary = folds.groupby('variant')[metrics].agg(['mean', 'std'])
display(summary.sort_values(('ROC-AUC', 'mean'), ascending=False).round(4))

## Результаты

![Сравнение моделей](../reports/figures/experiments/model_comparison.png)

CatBoost/base даёт AUC 0,7911. Исходные признаки дают 0,7865. Серии задержек и месячные отношения не дают устойчивого улучшения. Их в базовый набор не включаем.

Повтор с seed=17 даёт AUC 0,7899. Ниже сравниваем средние метрики обоих разбиений.

In [ ]:
repeat = pd.read_csv(root / 'reports/experiments/seed_17/fold_metrics.csv')
comparison = pd.concat([folds.assign(seed=42), repeat.assign(seed=17)])
keep = ['hgb_base', 'cat_base', 'cat_monthly', 'xgb_base']
display(comparison[comparison.variant.isin(keep)].groupby(
    ['variant', 'seed'])[metrics].mean().round(4))

## Проверка вероятностей

![Калибровка](../reports/figures/experiments/calibration.png)

Точки сравнивают средний прогноз и долю дефолта в десяти группах OOF. OOF содержит прогнозы строк моделями, которые на них не обучались.

Sigmoid проверен с тремя внутренними групповыми блоками в каждом из пяти внешних. Внешние ответы используются только для оценки. Средний Brier ухудшился с 0,13213 до 0,13294. Оставляем исходные вероятности.

In [ ]:
sigmoid = pd.read_csv(folder / 'sigmoid_metrics.csv').set_index('fold')
base = folds[folds.variant.eq('cat_base')].set_index('fold')
display(pd.DataFrame({'Brier до': base.Brier, 'Brier после': sigmoid.Brier}).round(5))

## Выбор модели

Используем **CatBoost/base**: 600 деревьев, глубина 6, learning_rate=0,04, l2_leaf_reg=5. Параметры и метрики сохранены в `reports/baseline.json`.

При проверке 20% клиентов охватываем 51,9% дефолтов. Среди выбранных дефолт у 57,1%. Это охват риска, а не оценка предотвращённых дефолтов.

При последнем коде −2, −1 или 0 AUC равен 0,6953. Раннее предупреждение даётся модели хуже. Коды −2 и 0 не подтверждают оплату вовремя.

CV использована для выбора кандидата. Итоговая оценка на test ещё не проведена. Рабочий порог требует стоимости ошибок.